# Flux Calibration & Stacking Demonstration

This notebook demonstrates the FITS calibration and image stacking pipeline using a variety of settings. We query raw frames for target **`ST9107`** observed on **`2013-11-06`**, reprocess them, align them using catalog-based alignment with global pointing consensus, stack them, and evaluate how background subtraction and calibration options affect the SNR of 10 selected catalog stars.

In [2]:
import sys
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
import astropy.units as u

# Dynamic path resolution to find workspace root
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
for path in [sys.path[0] if sys.path else '', os.getcwd()]:
    if not path: continue
    p = os.path.abspath(path)
    while p:
        if os.path.isdir(os.path.join(p, 'package')) and os.path.isfile(os.path.join(p, 'pytest.ini')):
            project_root = p
            break
        parent = os.path.dirname(p)
        if parent == p: break
        p = parent
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from package.src.astropipeline import astropipeline_etl as aple
from package.src.astropipeline import astropipeline_correct as aplc
from package.src.astropipeline import astropipeline_manager as aplmgr
from package.src.astropipeline import astropipeline_stack as apls
from package.src.astropipeline import astropipeline_measure as aplm

## 1. Retrieve and Reprocess Raw Frames

We query the NoirLab Adv Search API to retrieve the raw frames and calibration products corresponding to `ST9107` on `2013-11-06`, then run `correct_subpipe` to execute uniformity corrections.

In [3]:
# Configure output paths relative to notebooks directory
aplmgr.output_folder = '../../fits/'
aplmgr.study_output_path = '../../fits/apl_study_dover.csv'

# Find instcals for ST9107 on 2013-11-06
study = aple.PipeStudy(telescope="kp4m", instrument="newfirm", exposure=10, filter="KXs", max_returns=20)
df = study.find_instcals()
df_st = df[(df["OBJECT"] == "ST9107") & (df["caldat"] == "2013-11-06")].reset_index(drop=True)
print(f"Found {len(df_st)} frames of ST9107.")

# Reprocess frames (forces raw data pull and correction if not already on disk)
study_df_corrected = aplmgr.correct_subpipe(df_st.copy())

Found 3 frames of ST9107.
Study details saved to: ../../fits/apl_study_dover.csv


## 2. Rectify Frames Using Catalog Alignment

We align and warp the images using the catalog-based rectification method (`method="catalog"`). This leverages our global consensus offset voting algorithm across focal plane quadrants.

In [4]:
# Rectify frames with catalog alignment
study_df_rectified = aplmgr.undistort_subpipe(study_df_corrected.copy(), method="catalog", catalog="2MASS")
rectified_paths = study_df_rectified["rectified_path"].tolist()
print("Rectified paths:", rectified_paths)

Starting rectification for ../../fits/kp1852990_dover.fits.fz
Calculating global consensus pointing offset across all extensions...
Global pointing offset detected: dx=244.0, dy=488.0 pixels (based on 3 consensus matches)
Attempting rectilinear correction based on star catalog data.
Using global pointing offset prior: dx=244.0, dy=488.0 pixels
Rectification failed for extension 1: Insufficient matched celestial objects (2) to perform RANSAC SolvePnP correction.
Attempting rectilinear correction based on star catalog data.
Using global pointing offset prior: dx=244.0, dy=488.0 pixels
Rectification failed for extension 2: Insufficient matched celestial objects (3) to perform RANSAC SolvePnP correction.
Attempting rectilinear correction based on star catalog data.
Using global pointing offset prior: dx=244.0, dy=488.0 pixels
Rectification failed for extension 3: Insufficient matched celestial objects (0) to perform RANSAC SolvePnP correction.
Saved rectified image to: ../../fits/kp1852990

## 3. Query Catalog & Select 10 Stars in the Footprint

We query 2MASS stars in the exposure area. Since the stacked image covers chip 1 of the exposure, we filter the query results to select 10 stars that physically overlap with chip 1 boundaries and span the catalog brightness range.

In [5]:
row = study_df_rectified.iloc[0]
stars_df = aple.get_catalog_stars(row, catalog="2MASS")
stars_df = stars_df.dropna(subset=["Kmag"]).sort_values("Kmag")

# Open reference rectified image to filter stars inside chip footprint boundaries
hdul_ref = fits.open(rectified_paths[0])
ref_hdu = None
for hdu in hdul_ref:
    if isinstance(hdu.data, np.ndarray) and hdu.data.ndim == 2:
        ref_hdu = hdu
        break
if ref_hdu is None:
    ref_hdu = hdul_ref[0]
wcs_ref = WCS(ref_hdu.header)
h_shape = ref_hdu.data.shape
hdul_ref.close()

valid_stars = []
for _, star in stars_df.iterrows():
    c = SkyCoord(star['ra'], star['dec'], unit='deg')
    try:
        px, py = wcs_ref.world_to_pixel(c)
        if 50 <= px <= h_shape[1] - 50 and 50 <= py <= h_shape[0] - 50:
            valid_stars.append(star)
    except Exception:
        continue
        
valid_stars_df = pd.DataFrame(valid_stars)
print(f"Total catalog stars: {len(stars_df)}. Stars inside stacked chip 1: {len(valid_stars_df)}.")

# Select 10 stars spanning the range of brightness (Kmag)
indices = np.linspace(0, len(valid_stars_df) - 1, 10, dtype=int)
selected_stars = valid_stars_df.iloc[indices].reset_index(drop=True)
print("Selected 10 stars:")
print(selected_stars[["ra", "dec", "Kmag", "2MASS"]])

Total catalog stars: 442. Stars inside stacked chip 1: 311.
Selected 10 stars:
          ra        dec    Kmag             2MASS
0  53.046783  37.382065   6.001  03321122+3722554
1  52.988318  37.370304  11.520  03315719+3722130
2  52.927312  37.244209  12.830  03314255+3714391
3  53.055853  37.292236  13.645  03321340+3717320
4  52.940157  37.368675  14.171  03314563+3722072
5  52.997185  37.333874  14.571  03315932+3720019
6  53.053429  37.294853  14.932  03321282+3717414
7  53.042605  37.300003  15.388  03321022+3718000
8  53.017652  37.258942  15.617  03320423+3715321
9  52.995622  37.364273  15.984  03315894+3721513


## 4. Stack Under Different Settings

We run four stacking configurations (Runs A, B, C, D) using combinations of wavelet vs. polynomial background subtractions, pre vs. post calibration, and sigma clipping.

In [6]:
runs = {
    "Run_A_Post_Cal_Wavelet": {"bg_sub_method": "Wavelet Decomposition", "cal_stacked_flux": True, "cal_frames_flux": False, "sigma_clip": True, "filename": "stacked_run_a_demo.fits"},
    "Run_B_Pre_Cal_Wavelet": {"bg_sub_method": "Wavelet Decomposition", "cal_stacked_flux": False, "cal_frames_flux": True, "sigma_clip": True, "filename": "stacked_run_b_demo.fits"},
    "Run_C_Post_Cal_Poly": {"bg_sub_method": "Polynomial 7D", "cal_stacked_flux": True, "cal_frames_flux": False, "sigma_clip": True, "filename": "stacked_run_c_demo.fits"},
    "Run_D_Post_Cal_No_Clip": {"bg_sub_method": "Wavelet Decomposition", "cal_stacked_flux": True, "cal_frames_flux": False, "sigma_clip": False, "filename": "stacked_run_d_demo.fits"}
}

run_results = {}
for name, config in runs.items():
    _, dest_path, _ = apls.stack_images(
        image_paths=rectified_paths,
        method="median",
        sigma_clip=config["sigma_clip"],
        bg_sub_method=config["bg_sub_method"],
        cal_stacked_flux=config["cal_stacked_flux"],
        cal_frames_flux=config["cal_frames_flux"],
        catalog_stars_df=selected_stars,
        output_path="../../fits/",
        output_filename=config["filename"]
    )
    run_results[name] = dest_path
print("Finished Stacking runs.")

Finished Stacking runs.


## 5. Measure SNR for Selected Celestial Objects

We define a function to measure a star's signal-to-noise ratio (SNR) based on sub-pixel centroiding and local background statistics in the stack.

In [7]:
def measure_star_snr(data, wcs, ra, dec, levels=[2, 3, 4]):
    coord = SkyCoord(ra, dec, unit="deg")
    try:
        cutout = Cutout2D(data, coord, (30, 30), wcs=wcs, mode='trim')
        crop_data = cutout.data
    except Exception:
        return np.nan
        
    crop_data = np.nan_to_num(crop_data, nan=0.0)
    crop_data[crop_data < 0] = 0.0
    
    center_y, center_x = crop_data.shape[0] // 2, crop_data.shape[1] // 2
    best_snr = -np.inf
    
    # Always include the default raw peak (without DoG alignment shift) as a baseline candidate
    candidates = [(center_y, center_x)]
    
    # Add DoG alignment candidates for the specified levels
    for lvl in levels:
        crop_dog = aplm.dog_2d(crop_data, sigma_hi=lvl, sigma_lo=lvl+1, mode='reflect')
        try:
            dog_local = aplm.get_adjacent_pixels(crop_dog, (center_y, center_x), extent=[5, 5], remove_mid=False)
            local_max_y, local_max_x = np.unravel_index(np.argmax(dog_local), dog_local.shape)
            dog_max_y = center_y - 5 + local_max_y
            dog_max_x = center_x - 5 + local_max_x
            candidates.append((dog_max_y, dog_max_x))
        except Exception:
            pass
            
    # For each candidate, adjust guess on crop_data and measure SNR
    for cy, cx in candidates:
        adj_y, adj_x = aplm.adjust_guess_location(crop_data, cy, cx, [5, 5])
        try:
            sub_crop = aplm.get_adjacent_pixels(crop_data, (adj_y, adj_x), extent=[5, 5], remove_mid=False)
            border_stats = aplm.get_border_stats(sub_crop)
            peak_val = crop_data[adj_y, adj_x]
            
            if border_stats[1] > 0:
                snr = (peak_val - border_stats[0]) / border_stats[1]
            else:
                snr = 0.0
            
            if snr > best_snr:
                best_snr = snr
        except Exception:
            continue
            
    return best_snr if best_snr != -np.inf else np.nan

snr_data = {"Star_ID": [f"Star_{i+1}" for i in range(10)], "2MASS": selected_stars["2MASS"].tolist(), "Kmag": selected_stars["Kmag"].tolist()}
for name, path in run_results.items():
    hdul = fits.open(path)
    data = hdul[0].data
    wcs = WCS(hdul[0].header)
    hdul.close()
    
    snrs = []
    for _, star in selected_stars.iterrows():
        snr = measure_star_snr(data, wcs, star["ra"], star["dec"])
        snrs.append(snr)
    snr_data[name] = snrs

snr_df = pd.DataFrame(snr_data)
print("SNR Summary Table:")
print(snr_df)

SNR Summary Table:
   Star_ID             2MASS    Kmag  Run_A_Post_Cal_Wavelet  \
0   Star_1  03321122+3722554   6.001                4.614062   
1   Star_2  03315719+3722130  11.520                3.585998   
2   Star_3  03314255+3714391  12.830                8.530670   
3   Star_4  03321340+3717320  13.645                4.906187   
4   Star_5  03314563+3722072  14.171               10.293422   
5   Star_6  03315932+3720019  14.571                4.029278   
6   Star_7  03321282+3717414  14.932                5.856889   
7   Star_8  03321022+3718000  15.388                5.310521   
8   Star_9  03320423+3715321  15.617                6.603902   
9  Star_10  03315894+3721513  15.984                6.032795   

   Run_B_Pre_Cal_Wavelet  Run_C_Post_Cal_Poly  Run_D_Post_Cal_No_Clip  
0               4.917028             1.977272                2.959654  
1               6.604384             0.000000                3.803769  
2               6.814743             2.210814               

## 6. Generate Comparison Plots

We create bar charts grouped by star comparing:
- Pre-stack vs. Post-stack calibration SNR
- Wavelet vs. Polynomial background subtraction SNR

In [8]:
star_ids = snr_df["Star_ID"]
x = np.arange(len(star_ids))
width = 0.35

# Find actual columns (accounts for filename suffixes in stack_images output)
col_a = [c for c in snr_df.columns if "Run_A_Post_Cal_Wavelet" in c][0]
col_b = [c for c in snr_df.columns if "Run_B_Pre_Cal_Wavelet" in c][0]
col_c = [c for c in snr_df.columns if "Run_C_Post_Cal_Poly" in c][0]

# Plot 1: Pre vs Post-stack Calibration
plt.figure(figsize=(10, 6))
plt.bar(x - width/2, snr_df[col_b], width, label="Pre-stack Cal", color="orange")
plt.bar(x + width/2, snr_df[col_a], width, label="Post-stack Cal", color="blue")
avg_line = (snr_df[col_b] + snr_df[col_a]) / 2
plt.plot(x, avg_line, color="red", marker="o", label="Average SNR", linewidth=2)
plt.xticks(x, star_ids)
plt.xlabel("Celestial Objects")
plt.ylabel("SNR")
plt.title("Pre vs Post-stack Calibration SNR Comparison")
plt.legend()
plt.savefig("plot_pre_vs_post_cal.png", bbox_inches='tight')
plt.show()

# Plot 2: Wavelet vs Polynomial background subtraction
plt.figure(figsize=(10, 6))
plt.bar(x - width/2, snr_df[col_a], width, label="Wavelet bg", color="teal")
plt.bar(x + width/2, snr_df[col_c], width, label="Polynomial 7D bg", color="purple")
avg_line_bg = (snr_df[col_a] + snr_df[col_c]) / 2
plt.plot(x, avg_line_bg, color="red", marker="o", label="Average SNR", linewidth=2)
plt.xticks(x, star_ids)
plt.xlabel("Celestial Objects")
plt.ylabel("SNR")
plt.title("Wavelet vs Polynomial 7D Background Subtraction SNR Comparison")
plt.legend()
plt.savefig("plot_wavelet_vs_poly.png", bbox_inches='tight')
plt.show()

C:\Users\flann\AppData\Local\Temp\ipykernel_18900\1321192957.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\flann\AppData\Local\Temp\ipykernel_18900\1321192957.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Save Best Stacked Image

We identify the stacking run with the highest average SNR across our catalog stars, copy it as `best_stacked_image.fits`, and inject a descriptive header detailing all stacking and calibration settings.

In [9]:
col_d = [c for c in snr_df.columns if "Run_D_Post_Cal_No_Clip" in c][0]
avg_snrs = {
    "Run_A_Post_Cal_Wavelet": snr_df[col_a].mean(),
    "Run_B_Pre_Cal_Wavelet": snr_df[col_b].mean(),
    "Run_C_Post_Cal_Poly": snr_df[col_c].mean(),
    "Run_D_Post_Cal_No_Clip": snr_df[col_d].mean()
}
best_run = max(avg_snrs, key=avg_snrs.get)
print(f"Best Stacked Image is from: {best_run} with average SNR: {avg_snrs[best_run]:.4f}")

best_stacked_source = run_results[best_run]
best_stacked_dest = "best_stacked_image.fits"

hdul = fits.open(best_stacked_source)
best_data = hdul[0].data
best_header = hdul[0].header.copy()
hdul.close()

# Add descriptive header values
best_header["STACKMETHOD"] = ("median", "Image stacking method")
orig_key = best_run.replace("_demo", "")
best_header["SIGMACLIP"] = (runs[orig_key]["sigma_clip"], "Sigma clipping threshold applied")
best_header["BGSUB"] = (runs[orig_key]["bg_sub_method"], "Background subtraction method")
best_header["CALFRAME"] = (runs[orig_key]["cal_frames_flux"], "Calibration on individual frames")
best_header["CALSTACK"] = (runs[orig_key]["cal_stacked_flux"], "Calibration on final stacked image")
best_header["BESTRUN"] = (best_run, "Run name identifier")

avg_snr_val = avg_snrs[best_run]
if np.isnan(avg_snr_val):
    avg_snr_val = 0.0
best_header["AVG_SNR"] = (float(avg_snr_val), "Average SNR of selected catalog stars")

best_hdu = fits.PrimaryHDU(data=best_data, header=best_header)
best_hdul = fits.HDUList([best_hdu])
best_hdul.writeto(best_stacked_dest, overwrite=True, output_verify="ignore")
best_hdul.close()
print(f"Saved best stacked image to: {best_stacked_dest}")

Best Stacked Image is from: Run_B_Pre_Cal_Wavelet with average SNR: 10.3206
Saved best stacked image to: best_stacked_image.fits


## 8. Generate Zooms for Top 3 SNR Targets

We select the top 3 SNR stars in the best run and save a 3x3 arcminute cutout around each target, including crosshairs and circles at their center coordinates.

In [10]:
best_run_col = [c for c in snr_df.columns if best_run in c][0]
best_run_snrs = snr_df[best_run_col].tolist()
snrs_clean = [0.0 if np.isnan(s) else s for s in best_run_snrs]
top_indices = np.argsort(snrs_clean)[::-1][:3]

hdul = fits.open(best_stacked_dest)
best_data = hdul[0].data
best_wcs = WCS(hdul[0].header)
hdul.close()

# 3 arcminutes to pixel count conversion
pixel_scale_deg = np.mean(np.abs(best_wcs.pixel_scale_matrix.diagonal()))
arcmin_3_in_deg = 3.0 / 60.0
size_pixels = int(np.round(arcmin_3_in_deg / pixel_scale_deg))
print(f"Calculated 3x3 arcminutes size in pixels: {size_pixels}x{size_pixels}.")

for idx in top_indices:
    star = selected_stars.iloc[idx]
    star_name = star["2MASS"]
    ra_val = star["ra"]
    dec_val = star["dec"]
    star_snr = best_run_snrs[idx]
    
    coord = SkyCoord(ra_val, dec_val, unit="deg")
    try:
        # Center cutout on the actual peak instead of the catalog position
        c_x, c_y = best_wcs.world_to_pixel(coord)
        c_x_int, c_y_int = int(np.round(c_x)), int(np.round(c_y))
        adj_y, adj_x = aplm.adjust_guess_location(best_data, c_y_int, c_x_int, [15, 15])
        adj_coord = best_wcs.pixel_to_world(adj_x, adj_y)
        
        cutout = Cutout2D(best_data, adj_coord, (size_pixels, size_pixels), wcs=best_wcs, mode='trim')
        crop_data = cutout.data
        
        center_x, center_y = cutout.wcs.world_to_pixel(adj_coord)
        
        plt.figure(figsize=(6, 6))
        vmin = np.nanpercentile(crop_data, 1)
        vmax = np.nanpercentile(crop_data, 99.5)
        plt.imshow(crop_data, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        
        plt.axvline(center_x, color='red', linestyle='--', alpha=0.5)
        plt.axhline(center_y, color='red', linestyle='--', alpha=0.5)
        circle = plt.Circle((center_x, center_y), size_pixels // 20, color='red', fill=False, linewidth=2)
        plt.gca().add_patch(circle)
        
        plt.colorbar(label='Flux (Jy)' if best_header.get("BUNIT") == "Jy" else 'Counts')
        plt.title(f"Target: 2MASS J{star_name}\nSNR: {star_snr:.2f}, Kmag: {star['Kmag']:.2f}")
        
        out_png_name = f"zoom_2MASS_J{star_name}.png"
        plt.savefig(out_png_name, bbox_inches='tight')
        plt.close()
        print(f"Saved zoomed target to: {out_png_name}")
    except Exception as e:
        print(f"Failed to generate zoom for star {star_name}: {str(e)}")


Calculated 3x3 arcminutes size in pixels: 455x455.
Saved zoomed target to: zoom_2MASS_J03321282+3717414.png
Saved zoomed target to: zoom_2MASS_J03315932+3720019.png
Saved zoomed target to: zoom_2MASS_J03314563+3722072.png
